# 第 18 章互动实验：金模板、阈值与过检漏检

用合成的周期版图和注入缺陷，观察阈值 k、统计方法对检出和误报的影响。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "18_传统缺陷检测"
ASSIGNMENT = "18-1_金模板比对.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def pattern(h=96, w=96):
    y, x = np.mgrid[0:h, 0:w]
    return 40.0 + ((x // 4) % 2) * 80.0 + (((y // 16) % 2 == 0) & ((x // 16) % 2 == 0)) * 60.0

def scene(seed, n_goods=12, noise=3.0):
    rng = np.random.default_rng(seed)
    base = pattern()
    goods = base + rng.normal(scale=noise, size=(n_goods, 96, 96))
    test = base + rng.normal(scale=noise, size=(96, 96))
    truth = []
    for _ in range(5):
        cy, cx = rng.integers(8, 88, 2)
        amp = rng.choice([-1, 1]) * rng.uniform(12, 40)
        test[cy - 1:cy + 2, cx - 1:cx + 2] += amp
        truth.append((float(cy), float(cx)))
    return goods, test, truth

def run_or_hint(experiment):
    """运行一个实验函数；它用到的作业函数还没完成时，打印提示并返回 None。"""
    try:
        return experiment()
    except NotImplementedError as exc:
        print(f"先完成作业：{exc}")
        return None

goods, test, truth = scene(0)
res = attempt(mod.robust_golden, goods)
if res is not None:
    median, sigma = res
    z = np.abs(test - median) / sigma
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(test, cmap="gray"); ax[0].set_title("待检图")
    ax[1].imshow(sigma, cmap="viridis"); ax[1].set_title("逐像素 sigma")
    ax[2].imshow(z, cmap="magma", vmax=10); ax[2].set_title("z 图")
    for cy, cx in truth: ax[2].plot(cx, cy, "c+", ms=12)
    plt.show()

## 实验 1：k 与检出、误报

预测：k 从 3 增加到 9，检出率和误报数各怎么变？曲线的拐点在哪？

In [ ]:
ks = [3, 4, 5, 6, 7, 8, 9]
cap, fp = [], []
for k in ks:
    tp_all = n_all = fp_all = 0
    for s in range(10):
        g, t, tr = scene(100 + s)
        r = attempt(mod.robust_golden, g)
        if r is None: break
        found = attempt(mod.detect_defects, t, *r, k=k, min_area=3)
        if found is None: break
        m = attempt(mod.match_defects, [d["centroid"] for d in found], tr, 3.0)
        if m is None: break
        tp_all += m["tp"]; n_all += len(tr); fp_all += m["fp"]
    else:
        cap.append(tp_all / n_all); fp.append(fp_all / 10); continue
    break
if len(cap) == len(ks):
    fig, ax1 = plt.subplots(); ax2 = ax1.twinx()
    ax1.plot(ks, cap, "o-b"); ax1.set_ylabel("检出率", color="b")
    ax2.plot(ks, fp, "s-r"); ax2.set_ylabel("每张图误报", color="r")
    ax1.set_xlabel("k"); plt.title("阈值与检出、误报"); plt.show()

## 实验 2：良品里混进坏图

12 张良品里有 3 张在同一块 16×16 的区域整体偏亮 80。待检图除了原来的 5 个缺陷，在这块区域里还有一个 3×3、幅度 40 的缺陷。比较中位数加 MAD 和均值加标准差。

预测：两种方法各自能不能检出污染区域里的那个缺陷？检不出的那一种，主要是模板被拉偏了，还是 sigma 被撑大了？

In [ ]:
def contaminated(n_bad=3):
    g, t, tr = scene(7)
    g, t = g.copy(), t.copy()
    g[:n_bad, 40:56, 40:56] += 80
    t[47:50, 47:50] += 40
    inside = (48.0, 48.0)
    med, sig = mod.robust_golden(g)
    mean, std = g.mean(0), np.maximum(g.std(0), 1.0)
    rows = []
    for name, (c, s) in {"中位数+MAD": (med, sig), "均值+标准差": (mean, std)}.items():
        found = mod.detect_defects(t, c, s, k=6, min_area=3)
        centroids = [d["centroid"] for d in found]
        m = mod.match_defects(centroids, tr + [inside], 3.0)
        hit = mod.match_defects(centroids, [inside], 3.0)["tp"] == 1
        rows.append((name, m["tp"], len(tr) + 1, m["fp"], hit, s[40:56, 40:56].mean(), np.abs(c - pattern())[40:56, 40:56].mean()))
    return rows

rows = run_or_hint(contaminated)
if rows:
    for name, tp, n, fp_count, hit, sigma_in, bias_in in rows:
        print(f"{name}：检出 {tp}/{n}，误报 {fp_count}；污染区里的缺陷{'检出' if hit else '漏掉'}，污染区 sigma 均值 {sigma_in:.1f}（真实噪声 3），模板偏差均值 {bias_in:.1f}")
    print("想一想：把 contaminated(3) 换成 contaminated(5) 再跑，中位数加 MAD 还撑得住吗？（教程第 3.2 节）")

## 实验 3：良品张数与误报

这次图里没有任何缺陷，只有标准差为 3 的高斯噪声。用 N 张良品建模板，再拿一张新的良品算 z，统计 z > 5 的像素占多少。sigma 准确已知时，这个比例的理论值是 5.7e-7，一张 96×96 的图上平均不到 0.01 个像素。

预测：N 取 5、9、15、30、100 时，一张良品图上大约各有多少个像素超过 5？和理论值差几个数量级？均值加标准差的做法会更多还是更少？

In [ ]:
import math

def goods_count_experiment(k=5.0, counts=(5, 9, 15, 30, 100), repeats=10):
    rows = []
    for n in counts:
        over_mad = over_std = total = 0
        for s in range(repeats):
            rng = np.random.default_rng(1000 * n + s)
            goods = pattern() + rng.normal(scale=3.0, size=(n, 96, 96))
            fresh = pattern() + rng.normal(scale=3.0, size=(96, 96))
            median, sigma = mod.robust_golden(goods)
            over_mad += int((np.abs(fresh - median) / sigma > k).sum())
            over_std += int((np.abs(fresh - goods.mean(0)) / np.maximum(goods.std(0, ddof=1), 1.0) > k).sum())
            total += fresh.size
        rows.append((n, over_mad / total, over_std / total, over_mad / repeats))
    return rows

rows = run_or_hint(goods_count_experiment)
if rows:
    theory = math.erfc(5.0 / math.sqrt(2))
    for n, p_mad, p_std, per_image in rows:
        print(f"N={n:3d}：中位数+MAD 的单像素误报概率 {p_mad:.1e}（每张图约 {per_image:.1f} 个像素），均值+标准差 {p_std:.1e}，理论值 {theory:.1e}")
    ns = [r[0] for r in rows]
    plt.loglog(ns, [max(r[1], 1e-7) for r in rows], "o-", label="中位数 + MAD")
    plt.loglog(ns, [max(r[2], 1e-7) for r in rows], "s-", label="均值 + 标准差")
    plt.axhline(theory, color="g", ls="--", label="sigma 已知时的理论值")
    plt.xlabel("良品张数 N"); plt.ylabel("良品像素 z > 5 的概率"); plt.legend(); plt.show()
    print("想一想：N 很小时误报为什么多？把“k=5 对应百万分之一”写进报告错在哪？（教程第 3.3 节）")

## 实验 4：待检图整体变亮

照明或曝光的漂移会让待检图相对模板整体变亮。把一张良品图乘上 1.00 到 1.30 的增益，直接比对，统计亮区（灰度 180 的方块）里 z > 5 的像素比例。再做线性亮度归一化（把待检图拟合成 a × 模板 + b，换算回去再比），重做一遍。

预测：教程第 4 节给的门槛估算是 k × sigma / 亮度。代入 k = 5、sigma = 3、亮度 180，增益到多少时亮区大约一半像素报警？暗区（灰度 40）的门槛是多少？归一化之后还剩多少？

In [ ]:
def brightness_experiment(gains=np.arange(1.0, 1.301, 0.02)):
    rng = np.random.default_rng(11)
    goods = pattern() + rng.normal(scale=3.0, size=(30, 96, 96))
    fresh = pattern() + rng.normal(scale=3.0, size=(96, 96))
    median, sigma = mod.robust_golden(goods)
    bright, dark = pattern() > 150, pattern() < 60
    rows = []
    for g in gains:
        shifted = fresh * g
        a, b = np.polyfit(median.ravel(), shifted.ravel(), 1)
        z_raw = np.abs(shifted - median) / sigma
        z_fix = np.abs((shifted - b) / a - median) / sigma
        rows.append((g, (z_raw > 5)[bright].mean(), (z_raw > 5)[dark].mean(), (z_fix > 5)[bright].mean()))
    return np.array(rows)

rows = run_or_hint(brightness_experiment)
if rows is not None:
    pct = 100 * (rows[:, 0] - 1)
    plt.plot(pct, 100 * rows[:, 1], "o-", label="亮区（灰度 180），不归一化")
    plt.plot(pct, 100 * rows[:, 2], "s-", label="暗区（灰度 40），不归一化")
    plt.plot(pct, 100 * rows[:, 3], "^-", label="亮区，归一化之后")
    plt.axvline(100 * 5 * 3 / 180, color="gray", ls="--", label="k × sigma / 亮度（亮区）")
    plt.xlabel("待检图比模板整体亮 (%)"); plt.ylabel("z > 5 的像素比例 (%)"); plt.legend(); plt.show()
    half = pct[np.argmax(rows[:, 1] >= 0.5)] if (rows[:, 1] >= 0.5).any() else None
    print("亮区报警比例第一次超过一半时的增益：", f"亮 {half:.0f}%" if half is not None else "这个范围内没有超过一半")
    print("想一想：换一台噪声只有三分之一的相机，这条曲线往哪边移？噪声低反而更怕亮度漂移，为什么？（教程第 4 节）")

## 实验 5：匹配半径

检测结果固定不变，只改 `match_defects` 的 `radius`。真值坐标加了标准差 1.5 px 的随机偏移，模拟人工标注或复查设备给的坐标本身有误差。这次用弱一些的缺陷（幅度 6 到 15）、k = 4.5、min_area = 2，让漏检和误报都出现。

预测：半径从 1 加到 20，检出率怎么变？是一直涨，还是到某个值就平了？半径很大时多出来的“检出”是真的检出吗？

In [ ]:
def radius_experiment(radii=(1, 2, 3, 5, 8, 20), images=30):
    rng = np.random.default_rng(5)
    cases = []
    for s in range(images):
        base = pattern()
        goods = base + rng.normal(scale=3.0, size=(12, 96, 96))
        test_img = base + rng.normal(scale=3.0, size=(96, 96))
        truth = []
        for _ in range(5):
            cy, cx = rng.integers(8, 88, 2)
            test_img[cy - 1:cy + 2, cx - 1:cx + 2] += rng.choice([-1, 1]) * rng.uniform(6, 15)
            truth.append((float(cy), float(cx)))
        found = mod.detect_defects(test_img, *mod.robust_golden(goods), k=4.5, min_area=2)
        jittered = [(cy + rng.normal(scale=1.5), cx + rng.normal(scale=1.5)) for cy, cx in truth]
        cases.append(([d["centroid"] for d in found], truth, jittered))
    rows = []
    for radius in radii:
        tp = fp = n = 0
        for pred, _, jittered in cases:
            m = mod.match_defects(pred, jittered, radius)
            tp += m["tp"]; fp += m["fp"]; n += len(jittered)
        rows.append((radius, tp / n, fp / images))
    exact = sum(mod.match_defects(pred, truth, 3.0)["tp"] for pred, truth, _ in cases) / sum(len(t) for _, t, _ in cases)
    return rows, exact

result = run_or_hint(radius_experiment)
if result:
    rows, exact = result
    for radius, capture, fp_per_image in rows:
        print(f"半径 {radius:2d} px：检出率 {capture:.0%}，每张图误报 {fp_per_image:.1f}")
    print(f"真值坐标没有误差、半径 3 px 时的检出率：{exact:.0%}")
    fig, ax1 = plt.subplots(); ax2 = ax1.twinx()
    ax1.semilogx([r[0] for r in rows], [r[1] for r in rows], "o-b"); ax1.axhline(exact, color="b", ls=":")
    ax2.semilogx([r[0] for r in rows], [r[2] for r in rows], "s-r")
    ax1.set_xlabel("匹配半径 (px)"); ax1.set_ylabel("检出率", color="b"); ax2.set_ylabel("每张图误报", color="r")
    plt.title("检测结果不变，只改匹配半径"); plt.show()
    print("想一想：同一份检测结果能报出差这么多的检出率。半径太小和太大各错在哪？合理的半径由什么决定？（教程第 6.1 节）")

## 破坏实验与记录

- 实验 4 里让待检图整体亮 10%，误报出现在哪些区域？把 z 图画出来看。
- 把 `min_area` 设为 1 和 9，分别会丢掉什么？
- 实验 3 里把 `min_sigma` 改成 0.1 再跑一遍，N=5 时的误报怎么变？为什么？
- 把实验 1 的曲线和你选的工作点、实验 3 的那张表、实验 5 里你认为合理的半径写进 `实验记录.md`。